# 03 — Clustering utenti e prodotti

## Obiettivi didattici

1. Selezionare K via silhouette.
2. Confrontare KMeans vs GMM (almeno due approcci, come da PW).
3. Profilare i cluster trovati con z-score per feature.
4. Dare un'interpretazione semantica (label) a ciascun cluster.


In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from ecom_clustering.data import load_raw, filter_events_until
from ecom_clustering.features import compute_user_features, select_user_modeling_features
from ecom_clustering.clustering import (fit_kmeans, fit_gmm, select_k,
                                         compare_kmeans_vs_gmm, KMEANS_K_RANGE)
from ecom_clustering.evaluation import (cluster_size_plot,
                                         cluster_profile_table,
                                         cluster_profile_heatmap)
from ecom_clustering.config import DEFAULT_CONFIG
ecom = load_raw()
as_of = ecom.time_max - pd.Timedelta(days=30)
feats = compute_user_features(filter_events_until(ecom.events, as_of), as_of,
                              user_universe=ecom.users.user_id)
cols = select_user_modeling_features(feats)
scaler = StandardScaler().fit(feats[cols])
X = scaler.transform(feats[cols])
print(f'X.shape = {X.shape}')

## Selezione K via silhouette

In [ ]:
best_k, results = select_k(X, k_range=KMEANS_K_RANGE)
summary = pd.DataFrame([{'K': r.k, 'silhouette': r.silhouette,
                          'inertia': r.inertia} for r in results])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(summary.K, summary.silhouette, 'o-')
axes[0].set(title='Silhouette vs K', xlabel='K', ylabel='silhouette'); axes[0].grid(True, alpha=.3)
axes[1].plot(summary.K, summary.inertia, 'o-')
axes[1].set(title='Inertia vs K (elbow)', xlabel='K', ylabel='inertia'); axes[1].grid(True, alpha=.3)
fig.tight_layout(); plt.show()
print(f'Best K (silhouette): {best_k}')

## KMeans vs GMM

Il PW chiede esplicitamente di confrontare almeno due approcci.

In [ ]:
cmp_df = compare_kmeans_vs_gmm(X, k=best_k)
print(cmp_df.to_string(index=False))
print('\n→ KMeans solitamente vince per silhouette su feature standardizzate;\n      GMM utile se la covarianza tra feature è alta.')

## Profilo dei cluster

In [ ]:
result = fit_kmeans(X, k=best_k)
labels = result.labels
cluster_size_plot(labels, title=f'Distribuzione cluster (K={best_k})')
plt.show()
profile = cluster_profile_table(feats, labels, cols)
profile

In [ ]:
cluster_profile_heatmap(profile, title=f'Profilo cluster (K={best_k}) — z-score per feature')
plt.show()

## Interpretazione semantica

Leggendo l'heatmap, i cluster tipici emergono come:

- **Power buyers**: alta frequency, monetary, conversion_rate.
- **Browsers**: alti view, basso conversion_rate, alta varietà.
- **Cold users**: zero acquisti, recency saturata.
- **Returning customers**: attività recente alta vs storico.
- **Price-sensitive**: `price_sensitivity_ratio` < 1 (acquistano sotto media viewata).

Le label esatte dipendono dai dati specifici.